# Move the two-cube box task to Newton

Pick up the red cube, place it in the receiving box, then do the same with the blue cube. This is the same task used in Blog 2 and in the CPU/GPU comparison, for both the **SO-101** and **Seeed reBot DevArm**.

Newton supplies the model, state and control interfaces. `SolverMuJoCo` advances the rigid-body physics. The controller asks the robot to grasp and carry; contacts decide whether the object follows.

Run the complete reference first, then complete the small substep exercise. The earlier stacking example remains an optional exercise at the end.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess
import sys

PART3 = Path.cwd().resolve()
assert PART3.name == "part3" and (PART3 / "box_newton_demo.py").is_file()
REFERENCE = True
ROBOTS = ("so101", "rebot")
SCRIPT = PART3 / ("solutions/so101_newton_solution.py" if REFERENCE else "box_newton_exercise.py")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = PART3 / ".generated" / "box_teaching" / RUN_ID
print("Python:", sys.executable)
print("Implementation:", SCRIPT.relative_to(PART3))


## Build the same scene

The shared builder imports the complete MJCF box scene, including the table, two free cubes and five box pieces. Register MuJoCo custom attributes before import so authored contact and actuator properties survive conversion. Each cube keeps its 44 mm side length and 0.08 kg mass.

`BoxTask` provides the box geometry and ordered placement slots. `build_newton_builder(..., task_name="box")` constructs the same scene for the interactive lesson and the batched comparison. Its initial robot coordinates are established before physics starts; subsequent commands change only robot targets.


In [ ]:
import numpy as np
import warp as wp
import newton
from box_task import BoxTask
from robots import get_robot
from solutions.newton_scene_solution import build_newton_builder

wp.init()
for robot in ROBOTS:
    task = BoxTask(get_robot(robot))
    print(robot, "box interior:", task.box.lower, "to", task.box.upper)
    print("Placement order:", list(task.slots))
    print("Waypoints:", [waypoint.name for waypoint in task.sequence])


## Separate commands from simulation state

The online controller converts each Cartesian waypoint into joint targets using inverse kinematics. It computes one command at 50 Hz and holds it for 20 physics substeps. The lesson uses **2,000 control frames**, **40,000 integration steps** and **40 simulated seconds**.

Newton allocates two `State` buffers and one `Control`. The robot target indices come from joint names and `joint_target_q_start`, so free cube coordinates never become servo targets. `box_newton_demo.py` contains the complete path from online controller to `Control`.

The single rigid-body solver owns the integrated coordinates between steps. Its reset initializes those coordinates once; `update_data_interval=0` avoids copying them back from Newton before every step. The cloth/cable coupling lesson uses a separate synchronization policy because several solvers exchange state there.


In [ ]:
FPS, SUBSTEPS, FRAMES = 50, 20, 2000
DT = 1.0 / FPS / SUBSTEPS
assert np.isclose(DT, 0.001)
assert FRAMES * SUBSTEPS == 40000
assert np.isclose(FRAMES / FPS, 40.0)


## Complete one Newton substep

Open `box_newton_exercise.py`. Its `integrate_substep` method is the exercise:

```python
state.clear_forces()
self.solver.step(state, other, self.control, None, DT)
return other, state
```

`other` receives the result and becomes the next input. The solver performs MuJoCo contact detection on both CPU and CUDA in this task, so the contacts argument is `None`. Clear forces before stepping; apply any intended external forces after clearing.

The surrounding runner owns the 20-substep loop, recording and physical checks. On CUDA it captures the supported simulation work; online Python IK stays outside the graph. It reuses the same control allocation for each new command.

Set `REFERENCE=False` above after implementing the exercise, then restart and run the notebook. An unfinished exercise raises its explicit `NotImplementedError`; it is not counted as a successful or skipped run.


## Run both robots on CPU

The completed reference is selected by default. Each run must demonstrate opposing jaw contact, a full lift, airborne carry, commanded opening, detached placement and a full final second of settling. Both complete cubes must fit in the box and the gripper must withdraw. The report also checks the actual physics clock and integration count.

A report is saved even when physical validation fails. Read that failure before changing the controller or materials; do not relax the checks to make the run pass.


In [ ]:
def run_box(robot, device="cpu", viewer="null"):
    report = OUTPUT / f"{robot}_{device.replace(':', '')}_{viewer}.json"
    command = [sys.executable, str(SCRIPT), "--task", "box", "--robot", robot,
               "--device", device, "--viewer", viewer, "--num-frames", "2000",
               "--sim-substeps", "20", "--test", "--report", str(report)]
    subprocess.run(command, cwd=PART3, check=True, timeout=1800)
    result = json.loads(report.read_text())
    assert result["status"] == "passed" and result["capacity_passed"]
    assert result["validation"]["passed"] and result["recorded_physics_steps"] == 40000
    return result

cpu_results = {robot: run_box(robot) for robot in ROBOTS}
for robot, result in cpu_results.items():
    print(robot, result["status"], result["recorded_physics_steps"], "physics steps")


## Choose CUDA and inspect the motion

The optional cell below runs the same box reference on one selected GPU. These one-world runs include online IK and observation work; they verify the task and do not measure GPU throughput. The dedicated benchmark notebook compares replicated workloads.

For a window, use `viewer="gl"` with a fresh output path. For a portable scene recording, use `viewer="usd"`; the USD file is saved beside the report. Watch each cube leave the table in the jaws, descend to its own slot and remain in the box after the gripper opens.


In [ ]:
RUN_GPU = False
GPU_DEVICE = "cuda:0"
if RUN_GPU:
    assert wp.get_device(GPU_DEVICE).is_cuda
    gpu_results = {robot: run_box(robot, GPU_DEVICE) for robot in ROBOTS}
else:
    print("CUDA box task not run in this notebook session.")

# Optional motion inspection; each viewer choice writes its own report.
# run_box("so101", "cpu", viewer="gl")
# run_box("rebot", "cpu", viewer="usd")


## Optional earlier stacking exercise

`newton_scene.py` and `so101_newton.py` retain the original scene-building and stacking exercises. Their completed reference remains available with `solutions/so101_newton_solution.py --task stack --viewer null --num-frames 600 --test`.

That shorter task teaches the API with one placement. Its result does not validate the two-cube receiving-box task above.

## Next

Continue to [Clean the Table](03__clean_the_table.ipynb) to add a shirt, cable and explicit solver coupling. The rigid-body box task here uses one `SolverMuJoCo`; the next lesson combines that solver with `SolverVBD`.

Open [the CPU/GPU benchmark](05__migration_benchmark.ipynb) to run the same two-cube task at increasing environment counts with matching precomputed controls.
